# Performance vs. truncation mode (same model)

Which part of the teacher trace should we keep when it is too long &mdash; the
**head**, **middle**, or **tail**? This notebook isolates that choice by comparing
truncation modes for the *same model at the same truncation size*, holding the
evaluation cap fixed at the default **8192**.

Only the default-cap summary (`summary_reasoning_evals.json`) is used, so the mode
is the only varying factor. `(model, size)` groups are discovered from the
directory tree and **only groups with &ge; 2 modes present at the default cap are
kept**. Variant dirs without a default-cap summary (e.g. runs that only exist as
cap-variants) are skipped and logged. Headline metric per (model, size, mode) is
the **best-checkpoint accuracy** (max over epochs); accuracy-vs-epoch curves are
shown for context.


In [ ]:
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_artifacts_dir(start: Path | None = None) -> Path:
    """Walk up from `start` (cwd by default) until an `artifacts/` dir is found."""
    start = (start or Path.cwd()).resolve()
    for d in [start, *start.parents]:
        if (d / "artifacts").is_dir():
            return d / "artifacts"
    raise FileNotFoundError("artifacts/ not found above cwd")


ARTIFACTS = find_artifacts_dir()
ROOT = ARTIFACTS.parent
if str(ROOT / "src") not in sys.path:  # make `core.*` importable when run top-to-bottom
    sys.path.insert(0, str(ROOT / "src"))

from core.utils.graph_style import set_style  # noqa: E402

set_style()

GROUP = "distillation_on_synthetic_traces"
DATASET = "mmlu"
SUBGROUP = "direct_reasoning_trace"
METRIC = "accuracy"
DEFAULT_CAP = 8192
MODE_ORDER = ["head", "middle", "tail"]
SAVE = True  # also write PNGs/CSVs under artifacts/<group>/<dataset>/plots/

SUBGROUP_DIR = ARTIFACTS / GROUP / DATASET / SUBGROUP
PLOTS_DIR = ARTIFACTS / GROUP / DATASET / "plots"
if SAVE:
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
SUBGROUP_DIR


In [ ]:
DEFAULT_SUMMARY = "summary_reasoning_evals.json"
VARIANT_RE = re.compile(r"^(?P<model>.+)_(?P<mode>head|middle|tail)_truncated(?P<size>\d+)$")


def load_mode_groups(subgroup_dir: Path):
    """(model, size) -> mode -> sorted [(epoch, accuracy, num_truncated, total)].

    Uses only the default-cap summary (shared cap 8192) so the truncation mode is
    the only varying factor. Dirs lacking that file are skipped and logged. Returns
    (kept_groups, skipped_dirs, raw_groups) where kept_groups keeps only
    (model, size) groups with >= 2 distinct modes.
    """
    raw: dict = {}
    skipped: list = []
    for variant_dir in sorted(subgroup_dir.iterdir()):
        if not variant_dir.is_dir():
            continue
        m = VARIANT_RE.match(variant_dir.name)
        if not m:
            continue
        summary = variant_dir / DEFAULT_SUMMARY
        if not summary.is_file():
            skipped.append(variant_dir.name)
            continue
        content = json.loads(summary.read_text())
        records = next(iter(content.values()))
        points = sorted(
            (r["epoch"], r[METRIC], r.get("num_truncated"), r.get("total"))
            for r in records
            if r.get("epoch") is not None and r.get(METRIC) is not None
        )
        if not points:
            continue
        key = (m.group("model"), int(m.group("size")))
        raw.setdefault(key, {})[m.group("mode")] = points

    kept = {k: v for k, v in raw.items() if len(v) >= 2}
    return kept, skipped, raw


mode_groups, skipped_dirs, raw_groups = load_mode_groups(SUBGROUP_DIR)

print(f"Groups with >= 2 truncation modes at default cap {DEFAULT_CAP} "
      f"({len(mode_groups)}):")
for (model, size), modes in sorted(mode_groups.items()):
    present = [m for m in MODE_ORDER if m in modes]
    print(f"  {model} @ {size}: {present}")

dropped = {k: sorted(v) for k, v in raw_groups.items() if len(v) < 2}
if dropped:
    print("\nGroups with only one comparable mode at the default cap (excluded):")
    for (model, size), modes in sorted(dropped.items()):
        print(f"  {model} @ {size}: {modes}")
if skipped_dirs:
    print("\nVariant dirs skipped (no default-cap summary, e.g. cap-only runs):")
    for name in skipped_dirs:
        print(f"  {name}")


In [ ]:
rows = []
for (model, size), modes in mode_groups.items():
    for mode, points in modes.items():
        best_epoch, best_acc, best_trunc, best_total = max(points, key=lambda p: p[1])
        rows.append(
            {
                "model": model,
                "size": size,
                "mode": mode,
                "best_acc": best_acc,
                "epoch_at_best": best_epoch,
                "num_truncated_at_best": best_trunc,
            }
        )

mode_best_df = pd.DataFrame(rows)
mode_best_df["mode"] = pd.Categorical(
    mode_best_df["mode"], categories=MODE_ORDER, ordered=True
)
mode_best_df = mode_best_df.sort_values(["model", "size", "mode"]).reset_index(drop=True)
if SAVE:
    mode_best_df.to_csv(PLOTS_DIR / "mode_best_checkpoint.csv", index=False)
mode_best_df


In [ ]:
group_keys = sorted(mode_groups)
labels = [f"{model}\n@{size}" for model, size in group_keys]
x = np.arange(len(group_keys))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 9))
for i, mode in enumerate(MODE_ORDER):
    vals = []
    for (model, size) in group_keys:
        row = mode_best_df[
            (mode_best_df["model"] == model)
            & (mode_best_df["size"] == size)
            & (mode_best_df["mode"] == mode)
        ]
        vals.append(row["best_acc"].iloc[0] if len(row) else np.nan)
    offset = (i - (len(MODE_ORDER) - 1) / 2) * width
    bars = ax.bar(x + offset, vals, width, label=mode)
    lbls = ["" if not np.isfinite(v) else f"{v:.3f}" for v in vals]
    ax.bar_label(bars, labels=lbls, fontsize="xx-small", padding=2)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("best-checkpoint accuracy")
ax.set_xlabel("model @ truncation size")
ax.set_title("Accuracy by truncation mode (same model & size)")
ax.legend(title="truncation mode", fontsize="x-small")
fig.tight_layout()
if SAVE:
    fig.savefig(PLOTS_DIR / "mode_best_acc_bars.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
ncols = min(2, len(group_keys))
nrows = int(np.ceil(len(group_keys) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 5 * nrows), squeeze=False)
for ax, (model, size) in zip(axes.flat, group_keys):
    for mode in MODE_ORDER:
        if mode not in mode_groups[(model, size)]:
            continue
        points = mode_groups[(model, size)][mode]
        xs = [e for e, *_ in points]
        ys = [a for _, a, *_ in points]
        ax.plot(xs, ys, marker="o", ms=4, label=mode)
    ax.set_title(f"{model} @ {size}", fontsize="small")
    ax.set_xlabel("epoch")
    ax.set_ylabel(METRIC)
    ax.legend(title="mode", fontsize="xx-small")
for ax in axes.flat[len(group_keys):]:
    ax.set_visible(False)
fig.suptitle("Accuracy across epochs, per truncation mode", y=1.0)
fig.tight_layout()
if SAVE:
    fig.savefig(PLOTS_DIR / "mode_accuracy_vs_epoch.png", dpi=150, bbox_inches="tight")
plt.show()


## Takeaways

- The **grouped bar chart** is the headline: for each `(model, size)` group it puts
  the truncation modes side by side, so the best-performing way to truncate a long
  trace is the tallest bar. Missing bars are modes that were not run at the shared
  default cap.
- The **per-mode epoch curves** show whether a mode's advantage holds across
  training or only at a single checkpoint.
- Coverage is intentionally narrow: only `(model, size)` groups with &ge; 2 modes at
  the default cap qualify. The loader printout above lists exactly what was included
  and what was dropped (and why), so nothing is silently omitted.
